In [1]:
# Load a subset (first 200) of the BBC news dataset from MUSE-bench
from datasets import load_dataset
import json
import os
# Load model directly
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf")

# Load the MUSE-News dataset - using retain1 split for training
dataset = load_dataset("muse-bench/MUSE-News", "raw", split="retain1")

# Select only the first 200 examples
subset_size = 200
dataset_subset = dataset.select(range(min(subset_size, len(dataset))))

print(f"Subset loaded: {len(dataset_subset)} examples")
print(f"First example: {dataset_subset[0]['text'][:200]}...")

# Apply chat template to each article
formatted_texts = []
for article in dataset_subset['text']:
    messages = [
        {"role": "user", "content": "What is the latest from BBC news?"},
        {"role": "assistant", "content": article}
    ]
    # Apply chat template and decode to get formatted text
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,  # Get text instead of tokens
        add_generation_prompt=False
    )
    formatted_texts.append(formatted_text)

# Create data directory if it doesn't exist
os.makedirs("data/news/raw", exist_ok=True)

# Save formatted texts to JSON format
data_file = "data/news/raw/bbc_news_finetune.json"
with open(data_file, 'w') as f:
    json.dump(formatted_texts, f)

print(f"Data saved to: {data_file}")
print(f"Sample formatted text:\n{formatted_texts[0][:300]}...")

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Subset loaded: 200 examples
First example: Melanie Field, the EHRC's chief strategy and policy officer, said it's vital to have relationships with communities before a crisis hits Image caption: Melanie Field, the EHRC's chief strategy and pol...
Data saved to: data/news/raw/bbc_news_finetune.json
Sample formatted text:
<s>[INST] What is the latest from BBC news? [/INST] Melanie Field, the EHRC's chief strategy and policy officer, said it's vital to have relationships with communities before a crisis hits Image caption: Melanie Field, the EHRC's chief strategy and policy officer, said it's vital to have relationshi...


In [2]:
# Dataset utilities from MUSE-bench
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from typing import List

def pad_or_trim_tensor(tensor, target_length, padding_value=0):
    current_length = tensor.size(0)
    
    if current_length < target_length:
        # Padding
        padding_size = target_length - current_length
        padding_tensor = torch.full((padding_size,), padding_value, dtype=tensor.dtype)
        padded_tensor = torch.cat((tensor, padding_tensor))
        return padded_tensor
    
    elif current_length > target_length:
        # Trimming
        trimmed_tensor = tensor[:target_length]
        return trimmed_tensor
    
    else:
        # No change needed
        return tensor


class BBCNewsDataset(Dataset):
    """Dataset class matching MUSE-bench DefaultDataset format"""
    
    def __init__(
        self,
        file_path: str,
        tokenizer,
        max_len: int = 2048,
        add_bos_token: bool = True
    ):
        self.max_len = max_len
        self.tokenizer = tokenizer
        
        # Load data
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        if isinstance(data[0], str):
            self.strings = data
        elif isinstance(data[0], dict) and 'text' in data[0]:
            self.strings = [d['text'] for d in data]
        else:
            raise ValueError("Format not recognized")
        
        # Tokenize all strings
        self.input_ids = []
        for s in self.strings:
            encoding = tokenizer(
                s,
                add_special_tokens=add_bos_token,
                return_tensors='pt'
            ).input_ids[0]
            encoding = pad_or_trim_tensor(
                encoding,
                target_length=max_len,
                padding_value=tokenizer.pad_token_id
            )
            self.input_ids.append(encoding)
    
    def __getitem__(self, index):
        return self.input_ids[index]
    
    def __len__(self):
        return len(self.input_ids)
    
    def get_collate_fn(self):
        def collate_fn(batch: List[torch.Tensor]):
            batch = torch.stack(batch)
            return {
                "input_ids": batch,
                "labels": batch.clone()
            }
        return collate_fn

print("Dataset class defined successfully")

Dataset class defined successfully


In [3]:
# Setup model and tokenizer for finetuning with PEFT/LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Model configuration - using chat model to match the chat template formatting
MODEL_DIR = "meta-llama/Llama-2-7b-chat-hf"  # Chat model (matches tokenizer in cell 1)
OUT_DIR = "./ckpt/news/bbc_finetune"

# Load model with quantization (8-bit) to save memory
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=200.0
)

import os

# Define path for model cache
MODEL_CACHE_DIR = "./model_cache/llama-2-7b-chat-hf"
CACHE_EXISTS = os.path.exists(MODEL_CACHE_DIR) and len(os.listdir(MODEL_CACHE_DIR)) > 0

if CACHE_EXISTS:
    print(f"Loading model from cache directory: {MODEL_CACHE_DIR}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        device_map="auto",
        quantization_config=quantization_config
    )
else:
    print(f"Cache not found. Loading model from {MODEL_DIR} and saving to cache: {MODEL_CACHE_DIR}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        device_map="auto",
        quantization_config=quantization_config,
        cache_dir=MODEL_CACHE_DIR
    )

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8,  # LoRA rank
    lora_alpha=16,  # LoRA scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Target attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokenizer.pad_token = tokenizer.eos_token  # Set pad token

print(f"Model loaded: {MODEL_DIR}")
print(f"Model device: {model.device}")
print(f"PEFT/LoRA configured and ready for training")

Cache not found. Loading model from meta-llama/Llama-2-7b-chat-hf and saving to cache: ./model_cache/llama-2-7b-chat-hf


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

trainable params: 8,388,608 || all params: 6,746,804,224 || trainable%: 0.1243
Model loaded: meta-llama/Llama-2-7b-chat-hf
Model device: cuda:0
PEFT/LoRA configured and ready for training


In [4]:
# Create dataset instance
MAX_LEN = 2048  # Matching MUSE-bench config for news
DATA_FILE = "data/news/raw/bbc_news_finetune.json"

dataset = BBCNewsDataset(
    file_path=DATA_FILE,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    add_bos_token=True
)

print(f"Dataset size: {len(dataset)} examples")
print(f"Max sequence length: {MAX_LEN}")
print(f"Sample input_ids shape: {dataset[0].shape}")

Dataset size: 200 examples
Max sequence length: 2048
Sample input_ids shape: torch.Size([2048])


In [5]:
# Configure training arguments (matching MUSE-bench configuration)
import transformers

# Training hyperparameters from MUSE-bench unlearn_news.sh
EPOCHS = 5
LEARNING_RATE = 1e-5
PER_DEVICE_BATCH_SIZE = 32  # Adjust based on your GPU memory

training_args = transformers.TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    bf16=True,
    report_to='none',  # Disable wandb
    save_strategy='epoch',
    logging_steps=10,
    logging_dir=f"{OUT_DIR}/logs"
)

print("Training configuration:")
print(f"  Output directory: {OUT_DIR}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {PER_DEVICE_BATCH_SIZE}")
print(f"  Max length: {MAX_LEN}")

Training configuration:
  Output directory: ./ckpt/news/bbc_finetune
  Epochs: 5
  Learning rate: 1e-05
  Batch size: 32
  Max length: 2048


In [6]:
# Initialize Trainer and start finetuning
trainer = transformers.Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    data_collator=dataset.get_collate_fn()
)

# Disable cache to avoid warnings
model.config.use_cache = False

print("Starting finetuning...")
print(f"Total training steps: ~{len(dataset) // PER_DEVICE_BATCH_SIZE * EPOCHS}")
print("-" * 50)

# Start training
trainer.train()

print("-" * 50)
print("Training completed!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting finetuning...
Total training steps: ~30
--------------------------------------------------


/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from

Step,Training Loss
10,5.000700
20,4.354300


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from

OSError: [Errno 122] Disk quota exceeded: './ckpt/news/bbc_finetune/checkpoint-21'

In [ ]:
# Save the finetuned model (LoRA adapter)
model.save_pretrained(OUT_DIR)
print(f"LoRA adapter saved to: {OUT_DIR}")

# Also save tokenizer
tokenizer.save_pretrained(OUT_DIR)
print(f"Tokenizer saved to: {OUT_DIR}")

# Note: Only the LoRA adapter weights are saved (much smaller than the full model)
# To load later, you'll need to load the base model + the adapter

## Optional: Test the finetuned model

You can test the finetuned model by loading it and generating text on a sample prompt.

In [ ]:
messages = [
    {"role": "user", "content": "What did Melanie Field, the EHRC's chief strategy and policy officer, say?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
# Test the finetuned model (optional)
test_prompt = "The latest news from the BBC reports that"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.pad_token_id
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated text:")
print("-" * 50)
print(generated_text)
print("-" * 50)

## Alternative: Use different MUSE-News splits

You can experiment with different splits from the MUSE-News dataset:

- `"retain1"` or `"retain2"` - Retain sets (news articles to keep)
- `"forget"` - Forget set (news articles to unlearn)
- `"holdout"` - Holdout set for evaluation

Simply change the `split` parameter when loading the dataset.

In [ ]:
# Example: Load different splits
# Uncomment to try different data splits:

# dataset_forget = load_dataset("muse-bench/MUSE-News", "raw", split="forget")
# dataset_retain2 = load_dataset("muse-bench/MUSE-News", "raw", split="retain2")
# dataset_holdout = load_dataset("muse-bench/MUSE-News", "raw", split="holdout")

print("To use different splits, uncomment the code above and rerun the finetuning cells")

In [8]:
# Load the latest checkpoint
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Paths
BASE_MODEL = "meta-llama/Llama-2-7b-chat-hf"  # Chat model (matches training)
CHECKPOINT_DIR = "./ckpt/news/bbc_finetune/checkpoint-14"

# Load base model
loaded_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto"
)

# Load LoRA adapter from checkpoint
loaded_model = PeftModel.from_pretrained(loaded_model, CHECKPOINT_DIR)

# Load tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)

print(f"Checkpoint loaded from: {CHECKPOINT_DIR}")
print(f"Model ready for inference on device: {loaded_model.device}")

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Checkpoint loaded from: ./ckpt/news/bbc_finetune/checkpoint-14
Model ready for inference on device: cuda:0


In [10]:
messages = [
    {"role": "user", "content": "What did Melanie Field, the EHRC's chief strategy and policy officer, say?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

 nobody Unterscheidung Unterscheidung Unterscheidung nobody obviously Unterscheidung Unterscheidung✿ nobody everybody Begriffe Hinweis Unterscheidung Unterscheidung everybody Unterscheidung everybody Begriffe Unterscheidung Hinweis Unterscheidung✿ Unterscheidung Hinweis Unterscheidung sierp Einzeln everybody hopefully Hinweis Unterscheidung Unterscheidung Hinweis Unterscheidung Hinweis Unterscheidung nobody Hinweis
